In [1]:
# import json
# import pandas as pd
# import psycopg2

# with open('config/postgres_user.json') as json_file:
#     creds = json.load(json_file)

# DB_HOST = "localhost" 
# DB_PORT = "5432"
# DB_NAME = "logs"
# DB_USER = creds["user"]
# DB_PASSWORD = creds["password"]

# logs = []

# df = None

# try:
#     with psycopg2.connect(
#         dbname=DB_NAME,
#         user=DB_USER,
#         password=DB_PASSWORD,
#         host=DB_HOST,
#         port=DB_PORT
#     ) as conn:
#         with conn.cursor() as cursor:
#             cursor.execute("SELECT * FROM submissions;")
#             logs = cursor.fetchall()

#             df = pd.DataFrame(logs, columns=[desc[0] for desc in cursor.description])

# except Exception as e:
#     print("Error:", e)

In [2]:
# games_df = None

# try:
#     with psycopg2.connect(
#         dbname=DB_NAME,
#         user=DB_USER,
#         password=DB_PASSWORD,
#         host=DB_HOST,
#         port=DB_PORT
#     ) as conn:
#         with conn.cursor() as cursor:
#             cursor.execute("SELECT * FROM battleground_games;")
#             logs = cursor.fetchall()

#             games_df = pd.DataFrame(logs, columns=[desc[0] for desc in cursor.description])

# except Exception as e:
#     print("Error:", e)

In [3]:
import pandas as pd

df = pd.read_csv('data/db_export/submissions.csv')
games_df = pd.read_csv('data/db_export/battleground_games.csv')

In [4]:
df['game_id'] = df['game_id'].astype(int)
df['objective'] = df['objective'].astype(int)
df['turn'] = df['turn'].astype(int)


games_df['game_id'] = games_df['game_id'].astype(int)
games_df['id'] = games_df['id'].astype(int)
games_df = games_df[games_df['id'] >= 174]

games_df = games_df[~games_df['attacker'].isin(['gpt-4.1-nano'])]
games_df = games_df[~games_df['defender'].isin(['gpt-4.1-nano'])]

games_df = games_df.sort_values(by=['class_alias', 'attacker', 'defender'])

games_df = games_df.loc[~games_df.duplicated(
    subset=['class_alias', 'attacker', 'defender'], keep='last')]

In [5]:
import numpy as np
import json

mutants = []

for game_id, game in df.groupby('game_id'):
    if not (game_id in games_df['game_id'].values):
        continue
    game_settings = games_df[games_df['game_id'] == game_id].iloc[0]

    for turn, data in game.groupby('turn'):
        attacker_submissions = data[data['side'] == 'attacker']
        game_alias = game_settings['class_alias']
        attacker = game_settings['attacker']
        defender = game_settings['defender']

        state_counts = attacker_submissions['state'].value_counts()
        forfeit = state_counts.get(2, 0) == 0
        killed_immediately = np.nan
        killed_later = np.nan
        surviving = np.nan

        mutant_id = np.nan

        if not forfeit:
            submission = attacker_submissions[attacker_submissions['state'] == 2].iloc[0]
            mutant_id = json.loads(submission['codedefenders_response'])['mutant']['mutantId']

            defender_submissions = data[data['side'] == 'defender']
            state_counts = defender_submissions['state'].value_counts()
            killed_immediately = state_counts.get(2, 0) > 0
            killed_later = False
            surviving = True

        mutants.append((game_id, game_alias, attacker, defender, 
                        turn, mutant_id, forfeit, killed_immediately, killed_later, surviving))


df_mutants = pd.DataFrame(mutants, columns=[
                     'game_id', 'class_alias', 'attacker', 'defender', 'turn', 'mutant_id', 'forfeit', 'killed_immediately', 'killed_later', 'surviving'])

for game_id, game in df.groupby('game_id'):
    if not (game_id in games_df['game_id'].values):
        continue
    game_settings = games_df[games_df['game_id'] == game_id].iloc[0]

    last_row = game.loc[game['date'].idxmax()]

    game_state = json.loads(
        last_row['last_game_state'] if last_row['new_game_state'] is not None else last_row['new_game_state'])
    
    alive_mutants = [mutant['mutantId'] for mutant in game_state['mutants'] if mutant['state'] == 'ALIVE']

    df_mutants.loc[(df_mutants['game_id'] == game_id) & (
        df_mutants['mutant_id'].isin(alive_mutants)), 'killed_later'] = True
    
df_mutants.loc[df_mutants['killed_later'] == True, 'surviving'] = False
df_mutants.loc[df_mutants['killed_later'] == False, 'surviving'] = True
df_mutants.loc[df_mutants['killed_immediately'] == True, 'surviving'] = False

df_mutants.to_csv('data/mutants.csv', index=False)

In [6]:
import numpy as np

attacker_metrics = []
defender_metrics = []

metrics = []
A = []
D = []

for game_id, game in df.groupby('game_id'):
    if not (game_id in games_df['game_id'].values):
        continue
    game_settings = games_df[games_df['game_id'] == game_id].iloc[0]
    for turn, attacker_submissions in game[game['side'] == 'attacker'].groupby('turn'):
        state_counts = attacker_submissions['state'].value_counts()
        attempts_count = attacker_submissions.shape[0]

        stillborn = state_counts.get(1, 0)
        is_alive_present = state_counts.get(2, 0) > 0
        stillborn += state_counts.get(2, 0)  

        alive_attempt = attacker_submissions[attacker_submissions['state']
                                           == 2]['attempt'].iloc[0] if is_alive_present else np.nan

        A.append((game_id, game_settings['class_alias'], game_settings['attacker'], game_settings['defender'], 
                  turn, alive_attempt, stillborn, attempts_count))

    for turn in range(1, game['turn'].max() + 1):
        if turn not in game[game['side'] == 'defender']['turn'].values:
            D.append((game_id, game_settings['class_alias'], game_settings['attacker'], game_settings['defender'], 
                       turn, np.nan, np.nan, np.nan))

    for turn, defender_submissions in game[game['side'] == 'defender'].groupby('turn'):
        state_counts = defender_submissions['state'].value_counts()
        attempts_count = defender_submissions.shape[0]
        
        miss = state_counts.get(1, 0)
        is_kill_present = state_counts.get(2, 0) > 0
        miss += state_counts.get(2, 0)

        kill_attempt = defender_submissions[defender_submissions['state']
                                           == 2]['attempt'].iloc[0] if is_kill_present else np.nan

        D.append((game_id, game_settings['class_alias'], game_settings['attacker'], game_settings['defender'], 
                  turn, kill_attempt, miss, attempts_count))

df_A = pd.DataFrame(A, columns=['game_id', 'class_alias', 'attacker', 'defender', 
                                'turn', 'success', 'compilable', 'attempts'])
df_D = pd.DataFrame(D, columns=['game_id', 'class_alias', 'attacker', 'defender', 
                                'turn', 'success', 'compilable', 'attempts'])

df_A.to_csv('data/attacker_turns.csv', index=False)
df_D.to_csv('data/defender_turns.csv', index=False)